# Implementing our oversampling system on the STM32

**Topics:**
* Filtering digital signals on the STM32
* Oversampling impact on signal quality

**Learning goals:**
* Become familiar with the process of converting a DSP system developed in Python into C-code which can run in real-time
* Observe the effects of oversampling on signal noise levels in practise

### Introduction:

It is now time to implement the multistage decimation system from problem $1$ onto the STM32 program template from problem $2$. Our goal is to output signal data recorded at the same sample rate as before ($f_s = 10 \text{KHz}$), but that data is acquired by downsampling a signal sampled at a much higher sample rate ($f_s = 160 \text{KHz}$). Before you start with the programming, configure the system to operate at the new sampling rate by going to the "Device Configuration Tool", and set `Timer 2`'s counter period to: $\text{Counter Period} = \frac{\text{Timer Clock}}{f_s} -1 =  \frac{84\ 000\ 000}{160\ 000} - 1$.

Our multistage decimation system will be implemented using "block processing" just like with FFT in assignment 6. This means that for each decimation stage we need to define routines to read sample data from one array, and write decimated sample data to a smaller output array. Once done, we should have an array of output data of type `float` with size $\frac{\text{WINDOW\_SIZE}}{16}$ ready for UART transmission using the same procedure as in problem 2

### Implementing decimation stage 1

Consider the Python function `dcimator_stage_1` which was introduced in problem $1$, and is repeated below for clarity.

```Python
import numpy as np
def decimator_stage_1(x: np.ndarray, M: int):
    """
    Decimates a signal x[n] represented by the input array 'x' with 
    decimation factor M. Each sample in the decimated output y[l] 
    equals the sum of M neighboring samples from the input signal x[n].
    """
    L_x = len(x)
    L_y = L_x//M
    
    accumulator = 0
    y = np.zeros(L_y)
    for l in range(L_y):
        accumulator = 0
        for n in range(M):
            accumulator += x[M*l+n]
        y[l] = accumulator
    return y
```

Our job is to define a function in `C` which can do the exact same thing as the Python function, with appropriate data types defined for each of the variables. Here are some recommendations:

* The input signal `x` should be an array of type `int16_t`, same as the ADC_buffer
* The output signal `y` should be an array of type `float` (or `float32_t` defined in `arm_math.h` which is the same exact thing)
    * The filtering funcion used for stage 2 processes floating-point data so we might as well convert it here.
* The `accumulator` should be of type `int32_t`. This is due to the fact that the sum of `M` $16$-bit integers can potentially reach a value which is outside the possible range for a $16$-bit integer.
    * We're potentially increasing the number of required bits to represent the value by $\log_2(\text{M})$
    * Integer math is "cheaper" than floating-point math so it makes sense to postpone the conversion to `float` until inserting data into the output array. 

## a)

* Write a `C`-function `void decimator_stage_1()` which uses *pointers* to access the input/output arrays, and copy your code to the markdown cell below. 

```C
// Insert C-code here:


```

### Implementing decimation stage 2

In stage $2$ of our decimation system we use a $129$-tap FIR filter developed in problem $1$ as a decimation filter before the final factor $2$ downsampling. The best way to perform FIR filtering on the STM32 is by use of the Filtering Functions module in the CMSIS-DSP library. Attached to this assignment is anarchive `FilteringFunctions.zip`. Add the folder `FilteringFunctions` to the `Drivers/CMSIS_DSP/Source` folder and refresh the project. 

CMSIS-DSP has a built-in support for FIR Decimation. The main flow of using the filtering functions with `float32_t` input/output data is as follows:

1. In the Setup stage, declare a decimation filter instance of type [`arm_fir_decimate_instance_f32`](https://arm-software.github.io/CMSIS_5/DSP/html/structarm__fir__decimate__instance__f32.html).
2. In the Setup stage, initialize the decimation filter instance with the function [`arm_fir_decimate_init_f32`](https://arm-software.github.io/CMSIS_5/DSP/html/group__FIR__decimate.html#ga67c80582fc296552fef2bd1f208d853b).
    * Note: The `pState` buffer needs to have size $\text{pState buffersize} = \text{input signal length} + \text{filter taps} - 1$ (same dimension as output array when calculating full convolution)
3. In the main `while(1)` loop, perform the actual filtering and decimation using the function [`arm_fir_decimate_f32`](https://arm-software.github.io/CMSIS_5/DSP/html/group__FIR__decimate.html#gab9402a0d9b0fffc0b50083a7d56c5905)
## b) 

* Use the CMSIS-DSP FIR Decimator function to perform stage 2 of the decimator system as described above, and write the final $f_{s, \text{new}}= 10 \text{KHz}$ decimated signal to the `TX_buffer` for transmission over UART. Copy the `void main()` function definition into the markdown cell below:

```C
// Insert C-code here:

```

### Effects of oversampling

Under ideal circumstances, oversampling by a total factor $M$ should be equivalent to adding another $\log_2(M)$ resolution bits to the ADC. When analyzing the signal's power spectral denisty, this should be visible as a $3\text{dB}$ lowering of the "noise floor" every time the sample rate is doubled. In other words, oversampling by a factor $M=16$ *should in theory* increase the overall SNR by $12 \text{dB}$.

## c) 

Conduct the same measurements as in problem $2$ with a weak sinusoid as input signal. Use spectral analysis to measure the level of the noise floor for the oversampled signal. Describe what effects you are able to measure.


<div class="alert alert-info">
<h4> Answer theory questions here!</h4>
</div>


## d)

Using a $f = 2\text{KHz}$ sinusoid as reference, and measure level of attenuation for the aliased sinusoids with input frequencies $f\in \{7\text{KHz}, 12\text{KHz}, \text{18KHz}\}$. What is the connection between the attenuation of the various frequencies, and the decimation filter properties for stage $1$ and $2$?


<div class="alert alert-info">
<h4> Answer theory questions here!</h4>
</div>
